# ARC-AGI-2 walkthrough

ARC-AGI-2 is a static program-synthesis benchmark. We observe a few input/output demonstrations, infer a transformation, then apply it to unseen test input grids. There is no action-taking environment.

## 1. Load the official public dataset

Run `make setup` and `make data` from the repository root first.

In [ ]:
from pathlib import Path

from arc_agi2_fun.data import load_official_split
from arc_agi2_fun.visualize import plot_task
from arc_agi2_fun.registry import available_solvers, create_solver
from arc_agi2_fun.evaluation import evaluate_tasks
from arc_agi2_fun.search import rank_programs

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

DATA_DIR = ROOT / 'data' / 'ARC-AGI-2'
training = load_official_split(DATA_DIR, 'training')
evaluation = load_official_split(DATA_DIR, 'evaluation')
print('training tasks:', len(training))
print('evaluation tasks:', len(evaluation))
print('solvers:', available_solvers())

## 2. Inspect one training task

Use the training split for ordinary development. Keep the public evaluation split as an outer validation set.

In [ ]:
TASK_ID = next(iter(training))
task = training[TASK_ID]
plot_task(task)

## 3. Score candidate programs on the demonstrations

The current library is intentionally tiny. Exact pair matches are primary; cell accuracy is only a diagnostic tie-breaker.

In [ ]:
for candidate in rank_programs(task):
    print(candidate.program.name, candidate.exact_pairs, '/', candidate.total_pairs, candidate.mean_cell_accuracy)

## 4. The replaceable solver boundary

A solver receives the entire task and returns exactly two attempts for each test input. Future grammar/GP/MCTS systems should implement this same interface.

In [ ]:
solver = create_solver('primitive')
predictions = solver.solve_task(task)
predictions[0] if predictions else None

## 5. Objective local evaluation

On public tasks whose test outputs are known, the evaluator uses the competition rule: a test output is solved if attempt 1 OR attempt 2 exactly matches the target grid.

In [ ]:
report = evaluate_tasks(evaluation, solver, limit=10)
print(f'solved={report.solved_outputs}/{report.total_outputs}')
print(f'pass@2={report.accuracy:.3%}')

## 6. Where MCTS now belongs

For ARC-AGI-2, an MCTS state should be a **partial program**, not a game board. An MCTS action chooses a grammar production. Terminal nodes are executable programs. Reward evaluates program behavior on the demonstrations, ideally with complexity and generalization regularization.